In [ ]:
import pandas as pd
import os
from pathlib import Path

# Get all data files in the raw data folder
data_dir = Path("data/raw")
data_files = sorted(data_dir.glob("*.xls*"))

print(f"Found {len(data_files)} files in data/raw folder")
print("=" * 80)


In [ ]:
# List all files
from pathlib import Path


print("\nData files found:")
for i, file in enumerate[Path](data_files, 1):
    print(f"{i}. {file.name}")
    
print("\n" + "=" * 80)


In [ ]:
# Analyze DAT files (DAT20250713_1.xls format)
dat_files = [f for f in data_files if f.name.startswith('DAT')]
print(f"\nAnalyzing {len(dat_files)} DAT files...\n")

# Load first DAT file as sample with proper header parsing
sample_file = dat_files[0]
print(f"Sample file: {sample_file.name}")
print("=" * 80)

# Read header rows (row 0 = sensor group names, row 1 = measurement names)
df_raw = pd.read_excel(sample_file, engine='xlrd', header=None, nrows=2)
group_names = df_raw.iloc[0].values
measurement_names = df_raw.iloc[1].values

# Create proper column names by combining group and measurement with a dash
column_names = []
for group, measurement in zip(group_names, measurement_names):
    # Forward fill group names (they're only in the first column of each group)
    if pd.notna(group) and str(group).strip():
        current_group = str(group).strip()
    else:
        current_group = column_names[-1].split(' - ')[0] if column_names else 'Unknown'
    
    # Combine group and measurement with dash
    if pd.notna(measurement) and str(measurement).strip():
        column_names.append(f"{current_group} - {str(measurement).strip()}")
    else:
        # Fallback if measurement name is missing
        column_names.append(f"{current_group} - Column_{len(column_names)}")

# Read the actual data
df = pd.read_excel(sample_file, engine='xlrd', skiprows=2, names=column_names)

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


In [ ]:
# Display column names organized by category
print("\n" + "-" * 100)
print("COLUMN CATEGORIES:")
print("-" * 100)

col_categories = {
    'Date_Time': [col for col in column_names if 'Date_Time' in col],
    'M/E PORT (Main Engine Port)': [col for col in column_names if 'M/E PORT' in col],
    'M/E STBD (Main Engine Starboard)': [col for col in column_names if 'M/E STBD' in col],
    'Shipboard GPS': [col for col in column_names if 'GPS' in col],
    'Shipboard AIS': [col for col in column_names if 'AIS' in col],
    'LC-2 PORT': [col for col in column_names if 'LC-2 PORT' in col],
    'LC-2 STBD': [col for col in column_names if 'LC-2 STBD' in col],
    'Shaft PORT': [col for col in column_names if 'Shaft PORT' in col],
    'Shaft STBD': [col for col in column_names if 'Shaft STBD' in col]
}

for category, cols in col_categories.items():
    if cols:
        print(f"\n{category}: ({len(cols)} columns)")
        for col in cols:
            print(f"  - {col}")


In [ ]:
# Display first few rows
print("\n" + "-" * 100)
print("FIRST 5 ROWS:")
print("-" * 100)
pd.set_option('display.max_columns', 8)
pd.set_option('display.width', 120)
print(df.head())


In [ ]:
# Data types summary
print("\n" + "-" * 100)
print("DATA TYPES SUMMARY:")
print("-" * 100)
type_counts = df.dtypes.value_counts()
for dtype, count in type_counts.items():
    print(f"{dtype}: {count} columns")


In [ ]:
# Basic statistics
print("\n" + "-" * 100)
print("BASIC STATISTICS (Numeric columns):")
print("-" * 100)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
print(df[numeric_cols].describe().T.head(15))

# Missing values
print("\n" + "-" * 100)
print("MISSING VALUES:")
print("-" * 100)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
if len(missing_df) > 0:
    print(missing_df.head(10))
else:
    print("✓ No missing values found!")


In [ ]:
# Analyze structure across all DAT files
print("\n" + "=" * 100)
print("CHECKING STRUCTURE ACROSS ALL DAT FILES")
print("=" * 100)

dat_info = []
for file in dat_files[:5]:  # Sample first 5 for performance
    try:
        df_temp = pd.read_excel(file, engine='xlrd', skiprows=2)
        dat_info.append({
            'File': file.name,
            'Rows': df_temp.shape[0],
            'Columns': df_temp.shape[1]
        })
    except Exception as e:
        dat_info.append({
            'File': file.name,
            'Rows': 0,
            'Columns': 0,
            'Error': str(e)[:50]
        })

summary_df = pd.DataFrame(dat_info)
print("\nSample of DAT Files (first 5):")
print(summary_df.to_string(index=False))
if len(dat_files) > 5:
    print(f"\n... and {len(dat_files) - 5} more DAT files")


In [ ]:
# Analyze other files
print("\n" + "=" * 100)
print("ANALYZING OTHER FILES")
print("=" * 100)
other_files = [f for f in data_files if not f.name.startswith('DAT')]

for file in other_files:
    print(f"\n{'='*80}")
    print(f"File: {file.name}")
    print('='*80)
    try:
        df_other = pd.read_excel(file, engine='openpyxl')
        print(f"Shape: {df_other.shape[0]:,} rows × {df_other.shape[1]} columns")
        print(f"\nColumn names ({len(df_other.columns)}):")
        for i, col in enumerate(df_other.columns[:25], 1):
            print(f"{i:2d}. {col}")
        if len(df_other.columns) > 25:
            print(f"   ... and {len(df_other.columns) - 25} more columns")
        print(f"\nFirst 3 rows:")
        print(df_other.head(3))
    except Exception as e:
        print(f"Error reading file: {e}")


In [ ]:
# Comprehensive Summary
print("\n" + "=" * 100)
print("COMPREHENSIVE DATA SUMMARY")
print("=" * 100)

print(f"\nTotal files found: {len(data_files)}")
print(f"  - DAT files: {len(dat_files)}")
print(f"  - Other files: {len(other_files)}")

print(f"\nSample DAT file analysis ({sample_file.name}):")
print(f"  - Total rows: {df.shape[0]:,}")
print(f"  - Total columns: {df.shape[1]}")
print(f"  - Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  - Data types: {type_counts.to_dict()}")

print("\nKey findings:")
print("  - DAT files contain time-series data with 1-second intervals")
print("  - Data includes: Main Engine (PORT/STBD), GPS, AIS, LC-2 sensors, Shaft measurements")
print("  - Each DAT file covers approximately 12 hours of data (43,200 seconds)")

print("\n✓ Analysis complete!")


In [ ]:
# Print column names for first 2 datasets to verify the combined headers
print("\n" + "=" * 100)
print("COLUMN NAMES FOR FIRST 2 DATASETS (to verify combined headers)")
print("=" * 100)

for i, file in enumerate(dat_files[:2], 1):
    print(f"\n{'='*80}")
    print(f"Dataset {i}: {file.name}")
    print('='*80)
    
    # Read header rows
    df_raw = pd.read_excel(file, engine='xlrd', header=None, nrows=2)
    group_names = df_raw.iloc[0].values
    measurement_names = df_raw.iloc[1].values
    
    # Create column names
    col_names = []
    for group, measurement in zip(group_names, measurement_names):
        if pd.notna(group) and str(group).strip():
            current_group = str(group).strip()
        else:
            current_group = col_names[-1].split(' - ')[0] if col_names else 'Unknown'
        
        if pd.notna(measurement) and str(measurement).strip():
            col_names.append(f"{current_group} - {str(measurement).strip()}")
        else:
            col_names.append(f"{current_group} - Column_{len(col_names)}")
    
    # Print sensor grouped columns
    sensor_groups = {
    'M/E PORT': (0, 10),
    'M/E STBD': (11, 20),
    'Shipboard GPS': (21, 24),
    'Shipboard AIS': (25, 33),
    'LC-2 PORT': (34, 36),
    'LC-2 STBD': (37, 39),
    'Shaft PORT': (40, 47),
    'Shaft STBD': (48, 50)
}
    
    for group_name, (start_col, end_col) in sensor_groups.items():
        print(f"\n{group_name} (columns {start_col}-{end_col}):")
        for idx in range(start_col, min(end_col + 1, len(col_names))):
            print(f"  [{idx:2d}] {col_names[idx]}")


In [ ]:
# Helper function to combine headers from multi-row header
def get_combined_columns(file_path):
    """Read first two rows and combine them with a dash."""
    df_raw = pd.read_excel(file_path, engine='xlrd', header=None, nrows=2)
    group_names = df_raw.iloc[0].values
    measurement_names = df_raw.iloc[1].values
    
    column_names = []
    for group, measurement in zip(group_names, measurement_names):
        if pd.notna(group) and str(group).strip():
            current_group = str(group).strip()
        else:
            current_group = column_names[-1].split(' - ')[0] if column_names else 'Unknown'
        
        if pd.notna(measurement) and str(measurement).strip():
            column_names.append(f"{current_group} - {str(measurement).strip()}")
        else:
            column_names.append(f"{current_group} - Column_{len(column_names)}")
    
    return column_names

# Group DAT files by date
print("\n" + "=" * 100)
print("CONCATENATING DAILY DAT FILES")
print("=" * 100)

from collections import defaultdict

# Group files by date
date_groups = defaultdict(list)
for file in dat_files:
    if file.name.startswith('DAT'):
        # Extract date from filename (DATYYYYMMDD_N.xls)
        parts = file.stem.split('_')
        if len(parts) == 2:
            date_part = parts[0].replace('DAT', '')
            date_groups[date_part].append(file)

print(f"\nFound {len(date_groups)} dates with DAT files:")
for date, files in sorted(date_groups.items()):
    print(f"  {date}: {len([f for f in files if '_1' in f.name])} files (_1 and _2)")

print("\n" + "-" * 100)


In [ ]:
# Process each date and concatenate _1 and _2 files
concatenation_summary = []

for date in sorted(date_groups.keys()):
    files = date_groups[date]
    
    # Find _1 and _2 files for this date
    file_1 = next((f for f in files if '_1' in f.name), None)
    file_2 = next((f for f in files if '_2' in f.name), None)
    
    if file_1 is None or file_2 is None:
        print(f"\n⚠ Skipping date {date}: missing _1 or _2 file")
        continue
    
    print(f"\nProcessing date {date}:")
    print(f"  - File 1: {file_1.name}")
    print(f"  - File 2: {file_2.name}")
    
    try:
        # Get combined column names from the first file
        column_names = get_combined_columns(file_1)
        
        # Read both files with combined headers
        df1 = pd.read_excel(file_1, engine='xlrd', skiprows=2, names=column_names)
        df2 = pd.read_excel(file_2, engine='xlrd', skiprows=2, names=column_names)
        
        print(f"  - File 1: {df1.shape[0]:,} rows")
        print(f"  - File 2: {df2.shape[0]:,} rows")
        
        # Concatenate vertically
        df_combined = pd.concat([df1, df2], ignore_index=True)
        print(f"  - Combined: {df_combined.shape[0]:,} rows × {df_combined.shape[1]} columns")
        
        # Save to CSV
        output_file = data_dir / f"DAT{date}.csv"
        df_combined.to_csv(output_file, index=False)
        print(f"  - ✓ Saved to: {output_file.name}")
        
        # Store summary
        concatenation_summary.append({
            'Date': date,
            'File 1 Rows': df1.shape[0],
            'File 2 Rows': df2.shape[0],
            'Combined Rows': df_combined.shape[0],
            'Columns': df_combined.shape[1],
            'Output File': output_file.name
        })
        
    except Exception as e:
        print(f"  - ✗ Error processing {date}: {e}")
        concatenation_summary.append({
            'Date': date,
            'Error': str(e)[:50]
        })

print("\n" + "=" * 100)
print("CONCATENATION COMPLETE")
print("=" * 100)


In [ ]:
# Display summary of concatenated files
print("\nSUMMARY OF CONCATENATED FILES:")
print("=" * 100)

summary_df = pd.DataFrame(concatenation_summary)
print(summary_df.to_string(index=False))

print("\n" + "=" * 100)
print(f"✓ Successfully concatenated {len([s for s in concatenation_summary if 'Error' not in s])} date pairs")
print(f"✓ CSV files saved in: {data_dir}")
print("=" * 100)


In [ ]:
import pandas as pd

df_csv = pd.read_csv("data/raw/DAT20250713.csv")

# Display all columns and their types
for col, dtype in zip(df_csv.columns, df_csv.dtypes):
    print(f"{col}: {dtype}")


In [ ]:
import pandas as pd

# Load merged dataset
df = pd.read_csv("data/raw/DAT20250713.csv")

# Convert datetime if not already
df['Date_Time - Column_0'] = pd.to_datetime(df['Date_Time - Column_0'])

# Find the last row before noon
last_before_noon = df[df['Date_Time - Column_0'] <= '2025-07-13 11:59:59'].tail(1)

# Find the first row after noon
first_after_noon = df[df['Date_Time - Column_0'] >= '2025-07-13 12:00:00'].head(1)

print("Last record before 12:00:", last_before_noon)
print("\nFirst record at or after 12:00:", first_after_noon)


In [ ]:
df['time_diff'] = df['Date_Time - Column_0'].diff().dt.total_seconds()
print(df['time_diff'].value_counts())


In [ ]:
gap_rows = df[df['time_diff'] != 1.0]
print(gap_rows)


In [ ]:
print(df.columns)

In [ ]:
# Drop duplicate timestamp
df = df.drop_duplicates(subset='Date_Time - Column_0', keep='first')

# Optional: reindex to 1-second intervals and interpolate
df['Date_Time - Column_0'] = pd.to_datetime(df['Date_Time - Column_0'])
df = df.set_index('Date_Time - Column_0').asfreq('1s')
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].interpolate()


In [ ]:

gap_rows = df[df['time_diff'] != 1.0]
print(gap_rows)


In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("data/raw/DAT20250713.csv")

# Ensure Date_Time column is datetime
df['Date_Time - Column_0'] = pd.to_datetime(df['Date_Time - Column_0'])

# Set datetime as index for convenience
df = df.set_index('Date_Time - Column_0')

# Compute time differences between consecutive rows
df['time_diff'] = df.index.to_series().diff().dt.total_seconds()

# 1. Duplicated timestamps (time_diff == 0)
duplicates = df[df['time_diff'] == 0]

# 2. Missing seconds (time_diff > 1)
missing = df[df['time_diff'] > 1]

print("Duplicated timestamps:")
print(duplicates)

print("\nMissing seconds:")
print(missing)


In [ ]:
import pandas as pd
from pathlib import Path

# Folder containing merged DAT CSV files
data_dir = Path("data/raw")
output_dir = Path("data/processed")
output_dir.mkdir(exist_ok=True, parents=True)

# List all merged DAT CSVs
csv_files = list(data_dir.glob("DAT*.csv"))

for file_path in csv_files:
    print(f"Processing {file_path.name}...")
    
    # Load CSV
    df = pd.read_csv(file_path)
    
    # Ensure Date_Time column is datetime
    df['Date_Time - Column_0'] = pd.to_datetime(df['Date_Time - Column_0'], errors='coerce')
    
    # Drop rows with invalid or NaT datetimes
    df = df.dropna(subset=['Date_Time - Column_0'])
    
    # Set datetime as index
    df = df.set_index('Date_Time - Column_0')
    
    # Drop exact duplicate timestamps
    df = df[~df.index.duplicated(keep='first')]
    
    # Reindex to continuous 1-second frequency
    df = df.asfreq('1s')
    
    # Interpolate all numeric columns to fill missing rows
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    df[numeric_cols] = df[numeric_cols].interpolate(method='linear')
    
    # Optional: forward-fill integer columns that shouldn't be fractional
    int_cols = df.select_dtypes(include=['int64']).columns
    df[int_cols] = df[int_cols].round(0).astype('Int64')  # nullable integer type
    
    # Reset index to keep Date_Time as a column
    df = df.reset_index()
    
    # Save cleaned CSV
    out_file = output_dir / file_path.name
    df.to_csv(out_file, index=False, date_format='%Y-%m-%d %H:%M:%S', float_format='%.6f')
    
    # Report summary
    print(f"Saved cleaned file: {out_file.name}")
    print(f"  Total rows: {len(df)}")
    print(f"  Duplicates removed: {len(df[df.duplicated(subset=['Date_Time - Column_0'])])}")
    print(f"  Any missing seconds? {df['Date_Time - Column_0'].diff().dt.total_seconds().max() > 1}\n")
